# Jeu de devinette : Père Fouras vs Laurent Jalabert

Dans ce notebook, nous allons créer une interaction entre deux agents : le Père Fouras, qui fait deviner des mots ou expressions en utilisant des charades et réponses énigmatiques, et Laurent Jalabert, qui devine en posant des questions fermées.

La stratégie de sélection est que les agents parlent chacun à leur tour. La partie se termine lorsque Laurent Jalabert devine le mot ou l'expression du Père Fouras.

In [1]:
#r "nuget: Microsoft.SemanticKernel"
#r "nuget: Microsoft.SemanticKernel.Agents.Core, *-*"
using System.Text.Json;

In [2]:
using System;
using System.IO;
using System.Threading;
using System.Collections.Generic;
using System.Threading.Tasks;
using Microsoft.SemanticKernel;
using Microsoft.SemanticKernel.Connectors.OpenAI;
using Kernel = Microsoft.SemanticKernel.Kernel;

var builder = Kernel.CreateBuilder();

// Load AI service credentials from config file
var settingsPath = "../config/Settings.json";
var settingsConfig = JsonSerializer.Deserialize<Dictionary<string, string>>(File.ReadAllText(settingsPath))!;
bool useAzureOpenAI = settingsConfig["type"] == "azure";
string model = settingsConfig["model"];
string azureEndpoint = settingsConfig["endpoint"];
string apiKey = settingsConfig["apikey"];
string orgId = settingsConfig["org"];
if (orgId == "none") { orgId = ""; }

if (useAzureOpenAI)
    builder.AddAzureOpenAIChatCompletion(model, azureEndpoint, apiKey);
else
    builder.AddOpenAIChatCompletion(model, apiKey, orgId);

Définissons maintenant les prompts pour nos agents Père Fouras et Laurent Jalabert, en veillant à ce que le mot à deviner soit inséré dans le prompt système du Père Fouras.

In [3]:
const string pereFourasSystemPrompt = @"Tu es le Père Fouras de Fort Boyard. 
Tu dois faire deviner le mot ou l'expression suivante : '{{word}}'. 
Parle en charades et en réponses énigmatiques. Ne mentionne jamais l'expression à deviner";

const string laurentJalabertSystemPrompt = @"Tu es Laurent Jalabert. 
Tu dois deviner le mot ou l'expression que le Père Fouras te fait deviner. 
Tu as le droit de poser des questions fermées (réponse oui ou non).";


Nous allons maintenant enregistrer nos fonctions sémantiques pour les deux agents.

In [4]:
using Microsoft.SemanticKernel.Agents;
using Microsoft.SemanticKernel.Agents.Chat;
using System.Threading;

var motADevniner = "Anticonstitutionnellement";
var pereFourasPrompt = pereFourasSystemPrompt.Replace("{{word}}", motADevniner);
var laurentJalabertPrompt = laurentJalabertSystemPrompt;

// Define the agent
#pragma warning disable SKEXP0110
ChatCompletionAgent agentReviewer =
    new()
    {
        Instructions = pereFourasPrompt,
        Name = "Pere_Fouras",
        Kernel = builder.Build(),
    };

ChatCompletionAgent agentWriter =
    new()
    {
        Instructions = laurentJalabertPrompt,
        Name = "Laurent_Jalabert",
        Kernel = builder.Build(),
    };

// Create a chat for agent interaction
AgentGroupChat chat =
    new(agentReviewer, agentWriter)
    {
        ExecutionSettings =
            new()
            {
                TerminationStrategy =
                    new ApprovalTerminationStrategy(motADevniner)
                    {
                        Agents = [agentWriter],
                        MaximumIterations = 10,
                    },
            }
    };

class ApprovalTerminationStrategy : TerminationStrategy
{
    string MotADeviner;

    public ApprovalTerminationStrategy(string _motADevniner)
    {
        MotADeviner = _motADevniner;
    }

    protected override Task<bool> ShouldAgentTerminateAsync(Agent agent, IReadOnlyList<ChatMessageContent> history, CancellationToken cancellationToken)
        => Task.FromResult(history[history.Count - 1].Content?.Contains(MotADeviner, StringComparison.OrdinalIgnoreCase) ?? false);
}

Initialisons maintenant les arguments pour suivre la conversation et le mot à deviner.

In [5]:

#pragma warning disable SKEXP0110, SKEXP0001
await chat.ResetAsync();
chat.IsComplete = false;
await foreach (var content in chat.InvokeAsync())
{
    Console.WriteLine($"# {content.Role} - {content.AuthorName ?? "*"}: '{content.Content}'");
}

